## MLflow's Model Registry

In [2]:
from mlflow.tracking import MlflowClient


MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

### Interacting with the MLflow tracking server

The `MlflowClient` object allows us to interact with...
- an MLflow Tracking Server that creates and manages experiments and runs.
- an MLflow Registry Server that creates and manages registered models and model versions. 

To instantiate it we need to pass a tracking URI and/or a registry URI

In [14]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

client.search_experiments()

[<Experiment: artifact_location='/home/manwe_acer/mlops-zoomcamp/02-experiment-tracking/mlruns/3', creation_time=1792856516698, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1792856516698, lifecycle_stage='active', name='my-cool-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/home/manwe_acer/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1792773436544, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1792773436544, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>]

In [5]:
client.create_experiment(name="my-cool-experiment")

'3'

Let's check the latest versions for the experiment with id `1`...

In [27]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='2',
    filter_string="metrics.rmse < 7",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [28]:
runs

[<Run: data=<RunData: metrics={'best_iteration': 994.0,
  'rmse': 6.293781812728734,
  'stopped_iteration': 999.0,
  'validation-rmse': 6.293774504890245}, params={'custom_metric': 'None',
  'early_stopping_rounds': '50',
  'learning_rate': '0.09585355369315604',
  'max_depth': '30',
  'maximize': 'None',
  'min_child_weight': '1.060597050922164',
  'num_boost_round': '1000',
  'objective': 'reg:linear',
  'reg_alpha': '0.018060244040060163',
  'reg_lambda': '0.011658731377413597',
  'seed': '42',
  'verbose_eval': 'True'}, tags={'mlflow.runName': 'unleashed-shoat-24',
  'mlflow.source.name': 'duration-prediction.ipynb',
  'mlflow.source.type': 'NOTEBOOK',
  'mlflow.user': 'manwe_acer'}>, info=<RunInfo: artifact_uri='/home/manwe_acer/mlops-zoomcamp/02-experiment-tracking/mlruns/2/7e06caa2b03a41cf9ff4b55e7d2d2f37/artifacts', end_time=1792849923388, experiment_id='2', lifecycle_stage='active', run_id='7e06caa2b03a41cf9ff4b55e7d2d2f37', run_name='unleashed-shoat-24', start_time=1792849719

In [29]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 7e06caa2b03a41cf9ff4b55e7d2d2f37, rmse: 6.2938
run id: bae8c9d4b3a44d9a86596ae636792b04, rmse: 6.3297
run id: 850160e356584c46952e407e8b4c2ca6, rmse: 6.3973
run id: 482d1bfb17014bf7a52fbcc26d7008e3, rmse: 6.4080
run id: 2c4af424bbee447a8d1d4510bcf1419b, rmse: 6.5003


### Interacting with the Model Registry

In this section We will use the `MlflowClient` instance to:

1. Register a new version for the experiment `nyc-taxi-regressor`
2. Retrieve the latests versions of the model `nyc-taxi-regressor` and check that a new version `4` was created.
3. Transition the version `4` to "Staging" and adding annotations to it.

In [13]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [17]:
run_id = "7e06caa2b03a41cf9ff4b55e7d2d2f37"
model_uri = f"runs:/{run_id}/models_mlflow"

mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/10/24 19:24:52 WARNING mlflow.tracking._model_registry.fluent: Run with id 7e06caa2b03a41cf9ff4b55e7d2d2f37 has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-62a86aa967914d66845c750917d868db instead


Created version '6' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1792862693005, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1792862693005, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='7e06caa2b03a41cf9ff4b55e7d2d2f37', run_link=None, source='models:/m-62a86aa967914d66845c750917d868db', status='READY', status_message=None, tags={}, user_id=None, version=6, workspace='default'>

In [26]:
model_name = "nyc-taxi-regressor"
latest_versions = client.search_model_versions(f"name='{model_name}'")

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}, tag : {version.tags}, label : {version.aliases}")

version: 3, stage: None, tag : {'model': 'GBR'}, label : []
version: 2, stage: None, tag : {'model': 'RFR'}, label : []
version: 1, stage: None, tag : {'model': 'xgboost'}, label : []


In [ ]:
model_version = 2
new_stage = "Staging"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_2139841/4090915774.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=['staging'], creation_timestamp=1792862887203, current_stage='Staging', deployment_job_state=None, description='', last_updated_timestamp=1792863537221, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='72bd0063f94a4f1b82b6529f5af8c73a', run_link='', source='models:/m-053e15abac77484ea3c36a2d713eb85c', status='READY', status_message=None, tags={'model': 'RFR'}, user_id=None, version=2, workspace='default'>

In [31]:
client.set_registered_model_alias(
    name=model_name, 
    alias="bad",
    version=2
)

In [32]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"Random Forest Regressor is shit ! modified on {date}"
)

<ModelVersion: aliases=['bad', 'staging'], creation_timestamp=1792862887203, current_stage='Staging', deployment_job_state=None, description='Random Forest Regressor is shit ! modified on 2026-10-24', last_updated_timestamp=1792863972172, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='72bd0063f94a4f1b82b6529f5af8c73a', run_link='', source='models:/m-053e15abac77484ea3c36a2d713eb85c', status='READY', status_message=None, tags={'model': 'RFR'}, user_id=None, version=2, workspace='default'>

In [35]:
date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=1,
    description=f"Xgboosting the beeest ! modified on {date}"
)

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=3,
    description=f"Gradientboosting is alright. modified on {date}"
)

<ModelVersion: aliases=['challenger'], creation_timestamp=1792862898459, current_stage='None', deployment_job_state=None, description='Gradientboosting is alright. modified on 2026-10-24', last_updated_timestamp=1792864123467, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='ae36cd24c4754e5d96f37d532f812174', run_link='', source='models:/m-559088cf343e4ae59876f74766bac6b6', status='READY', status_message=None, tags={'model': 'GBR'}, user_id=None, version=3, workspace='default'>

### Comparing versions and selecting the new "Production" model

In the last section, we will retrieve models registered in the model registry and compare their performance on an unseen test set. The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:

1. Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2021.
2. Download the `DictVectorizer` that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
3. Preprocess the test set using the `DictVectorizer` so we can properly feed the regressors.
4. Make predictions on the test set using the model versions that are currently in the "Staging" and "Production" stages, and compare their performance.
5. Based on the results, update the "Production" model version accordingly.


**Note: the model registry doesn't actually deploy the model to production when you transition a model to the "Production" stage, it just assign a label to that model version. You should complement the registry with some CI/CD code that does the actual deployment.**

In [39]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd


def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = (df.lpep_dropoff_datetime - df.lpep_pickup_datetime).dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

def test_model_alias(name, alias, X_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}@{alias}")
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [40]:
df = read_dataframe("data/green_tripdata_2021-03.parquet")

In [41]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/home/manwe_acer/mlops-zoomcamp/02-experiment-tracking/preprocessor'

In [42]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [43]:
X_test = preprocess(df, dv)

In [44]:
target = "duration"
y_test = df[target].values

In [46]:
%time test_model_alias(name=model_name, alias="challenger", X_test=X_test, y_test=y_test)

CPU times: user 429 ms, sys: 144 ms, total: 574 ms
Wall time: 739 ms


{'rmse': 6.659623830022515}

In [48]:
%time test_model_alias(name=model_name, alias="production", X_test=X_test, y_test=y_test)

CPU times: user 1min 8s, sys: 402 ms, total: 1min 9s
Wall time: 12.7 s


{'rmse': 6.247320120758223}

In [43]:
client.transition_model_version_stage(
    name=model_name,
    version=4,
    stage="Production",
    archive_existing_versions=True
)

<ModelVersion: creation_timestamp=1652971637398, current_stage='Production', description='The model version 4 was transitioned to Staging on 2022-05-19', last_updated_timestamp=1652972763255, name='nyc-taxi-regressor', run_id='b8904012c84343b5bf8ee72aa8f0f402', run_link=None, source='./mlruns/1/b8904012c84343b5bf8ee72aa8f0f402/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=4>